# Pain diary cohort

Does treatment arm B change how pain evolves over the first six months? The data is synthetic, made for the Whybook demo.

## Load and reshape

In [1]:
import whybook
import pandas as pd
import statsmodels.formula.api as smf
from prep import MIN_DAYS, drop_sparse, load_diary_raw, load_olink, load_patients, to_long

In [2]:
diary_raw = load_diary_raw()
patients = load_patients()
olink = load_olink()

In [3]:
diary = to_long(diary_raw)
diary.head()

,patient_id,diary_day,week,pain_score,sleep_hours,analgesic_use,cycle_day,bleeding,mood,notes
0,P001,1,1,2.0,7.4,none,17,False,4.0,
1,P001,2,1,2.0,7.5,none,18,False,5.0,
2,P001,3,1,2.0,8.4,none,19,False,4.0,
3,P001,4,1,3.0,7.6,none,20,False,4.0,
4,P001,5,1,2.0,7.3,none,21,False,5.0,


## Shape

In [4]:
weekly = (
    diary.pipe(drop_sparse)
    .merge(patients[["patient_id", "treatment_arm"]], on="patient_id")
    .groupby(["patient_id", "treatment_arm", "week"], as_index=False, observed=True)["pain_score"]
    .mean()
)
whybook.ribbon(weekly, x="week", y="pain_score", by="treatment_arm", ci="bootstrap", n_boot=1000)

<whybook ribbon: pain_score by week>

## Model

In [5]:
model_data = weekly.merge(patients[["patient_id", "age", "site"]], on="patient_id", how="left").assign(
    month=lambda d: (d["week"] - 1) / 4.345
)
lmm_fit = smf.mixedlm(
    "pain_score ~ treatment_arm * month + age",
    data=model_data,
    groups="patient_id",
    re_formula="~month",
    missing="drop",
).fit()
lmm_fit.summary().tables[1]

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,5.009,0.355,14.113,0.000,4.313,5.705
treatment_arm[T.B],-0.857,0.160,-5.359,0.000,-1.170,-0.544
month,-0.750,0.036,-20.595,0.000,-0.821,-0.679
treatment_arm[T.B]:month,-0.318,0.050,-6.383,0.000,-0.416,-0.221
age,-0.003,0.009,-0.281,0.779,-0.021,0.016
patient_id Var,1.751,0.211,,,,
patient_id x month Cov,-0.250,0.051,,,,
month Var,0.152,0.020,,,,


In [7]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

ordinal_data = model_data.assign(
    pain_level=pd.cut(model_data["pain_score"], [-1, 3, 6, 10], labels=["mild", "moderate", "severe"]),
    arm_b=(model_data["treatment_arm"] == "B").astype(int),
)
estimates = []
for i in range(40):
    whybook.progress(i / 40, f"bootstrap {i + 1} of 40")
    sample = ordinal_data.sample(frac=1, replace=True, random_state=i)
    fit = OrderedModel.from_formula("pain_level ~ 0 + arm_b * month + age", sample, distr="logit").fit(method="bfgs", disp=False)
    estimates.append(fit.params["arm_b:month"])
whybook.progress(1.0, "done")
ordinal_arm_month = pd.Series(estimates, name="arm B × month (log odds)").describe()
ordinal_arm_month

count    40.000000
mean     -1.400307
std       0.083718
min      -1.632561
25%      -1.433042
50%      -1.403292
75%      -1.355648
max      -1.216267
Name: arm B × month (log odds), dtype: float64

In [6]:
model_data = model_data.assign(fitted=lmm_fit.fittedvalues, residual=lmm_fit.resid)
whybook.scatter(model_data, x="fitted", y="residual", by="treatment_arm", max_points=1500, title="Residuals against fitted values")
display(model_data.groupby("treatment_arm")[["residual"]].describe().round(3))
print(f"Residual SD {model_data['residual'].std():.2f}; {int((model_data['residual'].abs() > 3).sum())} residuals beyond 3")

<whybook scatter: Residuals against fitted values>

residual                                               
                 count mean    std    min    25%    50%    75%    max
treatment_arm                                                        
A               2737.0  0.0  0.633 -2.396 -0.435 -0.015  0.395  2.710
B               3100.0  0.0  0.785 -2.005 -0.585 -0.037  0.533  3.125

Residual SD 0.72; 3 residuals beyond 3


In [18]:
sweep = []
for min_days in [7, 14, 21]:
    whybook.progress(len(sweep) / 3, f"MIN_DAYS = {min_days}")
    kept = (
        diary.pipe(drop_sparse, min_days)
        .merge(patients[["patient_id", "treatment_arm", "age"]], on="patient_id")
        .groupby(["patient_id", "treatment_arm", "age", "week"], as_index=False, observed=True)["pain_score"]
        .mean()
        .assign(month=lambda d: (d["week"] - 1) / 4.345)
    )
    fit = smf.mixedlm("pain_score ~ treatment_arm * month + age", kept, groups="patient_id", re_formula="~month").fit()
    sweep.append(
        {
            "min_days": min_days,
            "patients": kept["patient_id"].nunique(),
            "arm_x_month": fit.params["treatment_arm[T.B]:month"],
            "p": fit.pvalues["treatment_arm[T.B]:month"],
        }
    )
whybook.progress(1.0, "done")
min_days_sweep = pd.DataFrame(sweep).round(4)
min_days_sweep

,min_days,patients,arm_x_month,p
0,7,307,-0.3273,0.0
1,14,291,-0.3183,0.0
2,21,291,-0.3183,0.0


In [16]:
no_east = model_data[model_data["site"] != "east"]
fit_no_east = smf.mixedlm("pain_score ~ treatment_arm * month + age", no_east, groups="patient_id", re_formula="~month").fit()
fit_no_east.summary().tables[1]

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,5.028,0.416,12.089,0.000,4.213,5.843
treatment_arm[T.B],-0.845,0.192,-4.409,0.000,-1.221,-0.470
month,-0.718,0.045,-15.894,0.000,-0.806,-0.629
treatment_arm[T.B]:month,-0.369,0.061,-6.032,0.000,-0.489,-0.249
age,-0.002,0.011,-0.144,0.886,-0.023,0.020
patient_id Var,1.888,0.259,,,,
patient_id x month Cov,-0.276,0.064,,,,
month Var,0.173,0.026,,,,


## Missing diary days

In [9]:
completeness = (
    diary.groupby("patient_id", as_index=False)
    .agg(days_logged=("diary_day", "size"), last_week=("week", "max"))
    .merge(patients[["patient_id", "treatment_arm", "site"]], on="patient_id")
    .assign(dropped=lambda d: d["last_week"] < 12)
)
display(completeness.groupby("treatment_arm")[["days_logged", "dropped"]].mean().round(2))
completeness

,days_logged,dropped
treatment_arm,,
A,108.29,0.26
B,124.04,0.09


,patient_id,days_logged,last_week,treatment_arm,site,dropped
0,P001,176,28,A,east,False
1,P002,152,25,A,east,False
2,P003,120,19,B,south,False
3,P004,147,23,B,east,False
4,P005,94,16,B,north,False
...,...,...,...,...,...,...
313,P314,140,22,B,west,False
314,P315,36,6,A,east,True
315,P316,39,7,A,west,True
316,P317,173,28,B,south,False


In [10]:
days_per_week = (
    diary.groupby(["patient_id", "week"], as_index=False)
    .size()
    .rename(columns={"size": "days"})
    .assign(share=lambda d: d["days"] / 7)
    .merge(patients[["patient_id", "treatment_arm"]], on="patient_id")
)
whybook.ribbon(days_per_week, x="week", y="share", by="treatment_arm", title="Share of diary days logged, by week")
days_per_week.groupby("treatment_arm")["share"].mean().round(3).to_frame("share logged")

<whybook ribbon: Share of diary days logged, by week>

,share logged
treatment_arm,
A,0.893
B,0.901


In [12]:
early_pain = (
    weekly[weekly["week"] <= 4].groupby("patient_id", as_index=False)["pain_score"].mean().rename(columns={"pain_score": "early_pain"})
)
dropout_data = completeness.merge(early_pain, on="patient_id").merge(patients[["patient_id", "age"]], on="patient_id")
dropout_fit = smf.logit("dropped ~ early_pain + treatment_arm + age", data=dropout_data.assign(dropped=lambda d: d["dropped"].astype(int))).fit(disp=False)
dropout_fit.summary2().tables[1].round(3)

/home/krassowski/.pyenv/versions/3.12.13/envs/future/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,-6.585,1.661,-3.965,0.000,-9.841,-3.330
treatment_arm[T.B],-23.296,20774.816,-0.001,0.999,-40741.187,40694.596
early_pain,1.014,0.246,4.115,0.000,0.531,1.497
age,-0.001,0.028,-0.026,0.980,-0.056,0.055


## Protein panel

In [13]:
import numpy as np

slopes = (
    weekly.groupby("patient_id")
    .apply(lambda d: np.polyfit(d["week"], d["pain_score"], 1)[0] if len(d) > 2 else np.nan, include_groups=False)
    .rename("pain_slope")
    .reset_index()
    .dropna()
    .merge(patients[["patient_id", "treatment_arm"]], on="patient_id")
)
whybook.hist(slopes, x="pain_slope", bins=40, title="Change in weekly pain per week, per patient")
slopes.groupby("treatment_arm")["pain_slope"].describe().round(3)

<whybook hist: Change in weekly pain per week, per patient>

,count,mean,std,min,25%,50%,75%,max
treatment_arm,,,,,,,,
A,147.0,-0.169,0.123,-0.544,-0.241,-0.178,-0.106,0.261
B,144.0,-0.249,0.094,-0.525,-0.314,-0.236,-0.182,-0.032


In [14]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

proteins = olink.set_index("patient_id")
joined = proteins.join(slopes.set_index("patient_id")["pain_slope"], how="inner")
names = list(proteins.columns)
r = joined[names].corrwith(joined["pain_slope"])
n = joined[names].notna().sum()
t = r * np.sqrt((n - 2) / (1 - r**2))
panel_of = {name: panel for panel, members in olink.attrs["whybook"]["groups"].items() for name in members}
protein_screen = (
    pd.DataFrame({"protein": names, "panel": [panel_of[name] for name in names], "n": n.to_numpy(), "r": r.to_numpy(), "p": 2 * stats.t.sf(np.abs(t), n - 2)})
    .assign(q=lambda d: multipletests(d["p"], method="fdr_bh")[1], neg_log10_p=lambda d: -np.log10(d["p"]))
    .sort_values("p")
    .reset_index(drop=True)
    .round({"r": 3, "p": 5, "q": 4, "neg_log10_p": 2})
)
whybook.scatter(protein_screen, x="r", y="neg_log10_p", by="panel", max_points=5000, title="Proteins against the change in pain")
display(protein_screen.head(10))
protein_screen

<whybook scatter: Proteins against the change in pain>

,protein,panel,n,r,p,q,neg_log10_p
0,NGF,Neurology,271,-0.281,0.00000,0.0129,5.57
1,CAR0600,Cardiometabolic,291,-0.200,0.00059,0.6225,3.23
2,IL6,Inflammation,264,-0.210,0.00059,0.6225,3.23
3,INF0627,Inflammation,284,0.202,0.00060,0.6225,3.22
4,ONC0935,Oncology,288,-0.198,0.00073,0.6225,3.14
5,CAR0246,Cardiometabolic,261,-0.204,0.00090,0.6225,3.04
6,INF0404,Inflammation,269,-0.201,0.00091,0.6225,3.04
7,NEU0015,Neurology,257,-0.201,0.00121,0.7110,2.92
8,INF0797,Inflammation,289,-0.185,0.00159,0.7110,2.80
9,CAR0842,Cardiometabolic,291,-0.182,0.00180,0.7110,2.74


,protein,panel,n,r,p,q,neg_log10_p
0,NGF,Neurology,271,-0.281,0.00000,0.0129,5.57
1,CAR0600,Cardiometabolic,291,-0.200,0.00059,0.6225,3.23
2,IL6,Inflammation,264,-0.210,0.00059,0.6225,3.23
3,INF0627,Inflammation,284,0.202,0.00060,0.6225,3.22
4,ONC0935,Oncology,288,-0.198,0.00073,0.6225,3.14
...,...,...,...,...,...,...,...
4807,ONC0290,Oncology,259,0.000,0.99940,0.9999,0.00
4808,ONC0987,Oncology,281,0.000,0.99942,0.9999,0.00
4809,ONC0844,Oncology,285,-0.000,0.99967,0.9999,0.00
4810,ONC0150,Oncology,271,0.000,0.99974,0.9999,0.00


In [15]:
hits_by_panel = (
    protein_screen.assign(hit=protein_screen["q"] < 0.05)
    .groupby("panel")
    .agg(proteins=("protein", "size"), hits=("hit", "sum"), strongest=("r", lambda s: s.abs().max()))
)
hits_by_panel

,proteins,hits,strongest
panel,,,
Cardiometabolic,1472,0,0.204
Inflammation,1104,0,0.210
Neurology,1180,1,0.281
Oncology,1056,0,0.198


## Secondary outcomes

In [16]:
sleep_weekly = (
    diary.merge(patients[["patient_id", "treatment_arm", "age"]], on="patient_id")
    .groupby(["patient_id", "treatment_arm", "age", "week"], as_index=False, observed=True)["sleep_hours"]
    .mean()
    .assign(month=lambda d: (d["week"] - 1) / 4.345)
)
whybook.ribbon(sleep_weekly, x="week", y="sleep_hours", by="treatment_arm", title="Weekly sleep hours, by arm")
sleep_fit = smf.mixedlm("sleep_hours ~ treatment_arm * month + age", sleep_weekly, groups="patient_id", missing="drop").fit()
sleep_fit.summary().tables[1]

<whybook ribbon: Weekly sleep hours, by arm>

/tmp/ipykernel_2/667835809.py:8: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  sleep_fit = smf.mixedlm("sleep_hours ~ treatment_arm * month + age", sleep_weekly, groups="patient_id", missing="drop").fit()


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,7.064,0.033,215.189,0.000,7.000,7.129
treatment_arm[T.B],-0.041,0.021,-1.933,0.053,-0.083,0.001
month,-0.006,0.005,-1.085,0.278,-0.016,0.005
treatment_arm[T.B]:month,0.014,0.007,1.815,0.070,-0.001,0.028
age,-0.001,0.001,-1.203,0.229,-0.003,0.001
patient_id Var,0.000,0.002,,,,


In [17]:
took_analgesic = diary.merge(patients[["patient_id", "treatment_arm"]], on="patient_id").assign(
    month=lambda d: ((d["week"] - 1) // 4 + 1).clip(upper=6),
    took=lambda d: (d["analgesic_use"] != "none").astype(float),
)
whybook.bars(took_analgesic, x="treatment_arm", y="took", title="Share of days with an analgesic, by arm")
analgesic_share = took_analgesic.pivot_table(index="month", columns="treatment_arm", values="took", aggfunc="mean", observed=True).round(3)
analgesic_share

<whybook bars: Share of days with an analgesic, by arm>

treatment_arm,A,B
month,,
1,0.472,0.466
2,0.504,0.471
3,0.476,0.496
4,0.482,0.487
5,0.503,0.501
6,0.509,0.500


## Report

In [18]:
table_one = pd.concat(
    [
        patients.groupby("treatment_arm").size().rename("patients").to_frame().T,
        patients.groupby("treatment_arm")[["age", "bmi"]].mean().round(1).T.rename(index=lambda name: f"{name}, mean"),
        patients.groupby("treatment_arm")["bmi"].apply(lambda s: s.isna().mean()).round(2).to_frame("bmi, share missing").T,
        pd.crosstab(patients["site"], patients["treatment_arm"]),
        pd.crosstab(patients["stage"], patients["treatment_arm"]),
    ]
)
table_one

treatment_arm,A,B
patients,159.00,159.00
"age, mean",36.30,37.00
"bmi, mean",26.30,26.90
"bmi, share missing",0.06,0.04
east,42.00,35.00
north,48.00,55.00
south,43.00,44.00
west,26.00,25.00
I,53.00,51.00
II,41.00,51.00


In [19]:
term = "treatment_arm[T.B]:month"
effects = pd.DataFrame(
    [
        {"analysis": "Mixed model", "estimate": lmm_fit.params[term], "p": lmm_fit.pvalues[term]},
        {"analysis": "Without site east", "estimate": fit_no_east.params[term], "p": fit_no_east.pvalues[term]},
        *({"analysis": f"MIN_DAYS = {row.min_days}", "estimate": row.arm_x_month, "p": row.p} for row in min_days_sweep.itertuples()),
    ]
).round(4)
whybook.bars(effects, x="analysis", y="estimate", title="Arm B × month, by analysis")
effects

<whybook bars: Arm B × month, by analysis>

,analysis,estimate,p
0,Mixed model,-0.3183,0.0
1,Without site east,-0.3688,0.0
2,MIN_DAYS = 7,-0.3273,0.0
3,MIN_DAYS = 14,-0.3183,0.0
4,MIN_DAYS = 21,-0.3183,0.0


## Summary

Pain falls over the six months in both arms, faster in arm B, by 0.32 points more per month, and the difference holds without site east and for each value of MIN_DAYS. More patients stop keeping the diary in the first 12 weeks in arm A (26%) than in arm B (9%), so the patients who stay are not a random half of either arm. Of the 4,812 proteins, only NGF tracks how fast pain changes once the false discovery rate is controlled.